# 工具系统与 MCP（Harness Engineering 学习笔记）

> 素材来源：《Harness Engineering 实战：构建可靠的生产级 AI Agent》第 4 章，配套代码仓库 [harness-py-book](https://github.com/Luxuzhou/harness-py-book)；文内代码为免 API Key 的可运行简化复现，对应入口 `python -B examples/ch04_tools.py`。

**先记住一句话：模型只面对"工具描述"，执行全部收口在工具层——描述决定模型会不会用，契约决定用得安不安全。**

| 事实 | 推论 |
|---|---|
| 表 4-3：v2 工具描述把违规调用率从 13.0% 压到 5.3% | 描述工程本身就是安全工程 |
| 表 4-4：准确率随描述长度呈"倒U"，303–521 Token 双峰 97.1% | 描述不是越长越好，过约束会反噬 |
| edit_positive 首次命中率 0%，任务内命中率却有 73.3%–83.3% | 单一指标必然误导，必须双轨报告 |
| v2 描述 + v2 提示词：bash 首次命中率 87.5% → 54.2% | 局部最优叠加可能负协同，要整体评测 |
| MCP 用 JSON-RPC 2.0 统一工具接入协议 | 工具生态与 Agent 彻底解耦 |

## 1. 工具层三层架构：模型只面对"描述"

第 4 章的第一个设计决策：把工具系统拆成三层，职责互不越界。

| 层 | 职责 | 关键点 |
|---|---|---|
| ① 模型层 | 只读工具描述、发起调用 | 看不到实现，也无法绕过注册表 |
| ② 注册表层 | **生产环境真正的安全边界** | `TOOL_SCHEMAS` 白名单、`get_schemas_for_phase` 阶段权限、`execute_tool` 三步收口 |
| ③ 实现通道层 | 工具真正落地 | 通道 A：本地 `TOOL_REGISTRY`（6 函数，零网络依赖）；通道 B：MCP Server |

<div style="text-align:center"><img src="../src/tool_layer_architecture.svg" width="820"/></div>

`execute_tool` 的三步收口是全章骨架：白名单校验 → 阶段权限 → try/except 全量兜底，任何出口都统一为 `tuple[bool, str]`。

In [ ]:
# ===== Demo 1：注册表层——execute_tool 的三步收口 =====
"""模型只看 TOOL_SCHEMAS 里的"描述"；真正执行前必须过三道闸。
对应书 4.2：白名单校验 → 阶段权限 → 全量兜底，出口统一 tuple[bool, str]。
"""
from dataclasses import dataclass, field

TOOL_SCHEMAS = {
    "read_file":  {"description": "读取项目内文本文件（自动解码、超长截断）", "readonly": True},
    "write_file": {"description": "写入文件（禁止项目根之外的路径）", "readonly": False},
    "bash":       {"description": "在项目内执行 shell 命令", "readonly": False},
}

@dataclass
class PhasePolicy:
    """阶段权限：规划期只读，执行期才放开写操作。"""
    phase: str = "plan"   # plan | act
    allowed: dict = field(default_factory=lambda: {
        "plan": {"read_file", "grep_search", "glob_search"},
        "act":  {"read_file", "write_file", "edit_file", "grep_search", "glob_search", "bash"},
    })

def execute_tool(name: str, args: dict, policy: PhasePolicy) -> tuple[bool, str]:
    # ① 白名单校验：未注册的工具直接拒绝
    if name not in TOOL_SCHEMAS:
        return False, f"未知工具 {name}，可用工具：{sorted(TOOL_SCHEMAS)}"
    # ② 阶段权限：规划期只读
    if name not in policy.allowed[policy.phase]:
        return False, f"当前处于 {policy.phase} 阶段，工具 {name} 未授权"
    # ③ 全量兜底：任何异常都转为可读错误，绝不让 Agent 崩溃
    try:
        return True, f"[{name}] 已执行（参数：{args}）"
    except Exception as e:
        return False, f"工具 {name} 执行失败：{e}"

policy = PhasePolicy(phase="plan")
print(execute_tool("write_file", {"path": "a.txt", "content": "hi"}, policy))
print(execute_tool("run_sql", {"sql": "drop table"}, policy))
policy.phase = "act"
print(execute_tool("write_file", {"path": "a.txt", "content": "hi"}, policy))

## 2. 一份统一契约：`tuple[bool, str]`，错误是反馈信号

6 个基础工具（read_file / write_file / edit_file / grep_search / glob_search / bash）共用同一签名：

```python
def tool_xxx(args: dict, config) -> tuple[bool, str]
```

- 成功：`(True, "结果内容")`；失败：`(False, "可读错误信息")`
- 错误信息是写给模型看的"下一轮反馈信号"，不是抛给人看的异常
- 四处防呆：路径前缀碰撞 / edit 唯一匹配 / grep 与 glob 返回格式刻意区分（命名唯一零同义词）/ bash 黑名单 + 后台线程超时强杀

<div style="text-align:center"><img src="../src/tools_contract.svg" width="820"/></div>

**4.5.2 新增工具的两条件**：①现有工具组合无法完成（能力闭包之外）；②多次调用且高频出错（书中 bash 误用率已从 15% 降到 3% 以下）。不满足就别加——工具越多，模型选错的概率越大。

**防呆一：write_file 的路径前缀碰撞。** `/project` 与 `/project_secret` 只差一个后缀，用 `startswith` 做白名单会被 `../project_secret` 绕过——必须先 `resolve()` 消除 `..` 与符号链接，再用 `is_relative_to()` 判定（书 4.3）。

In [ ]:
# ===== Demo 2：write_file 防呆——resolve() 挡住路径前缀碰撞 =====
"""/project 与 /project_secret 只差一个后缀（书 4.3）：
startswith 白名单会被 ../project_secret 绕过，
必须先 resolve() 再 is_relative_to()。
"""
from pathlib import Path

PROJECT = Path("/tmp/project")           # 项目根
SECRET = Path("/tmp/project_secret")     # 同级"影子"目录（含敏感配置）

evil = (PROJECT / "../project_secret/config.ini").resolve()
print("startswith 判定（误判为安全）:", str(evil).startswith(str(PROJECT.resolve())))

def check_path(path: str) -> tuple[bool, str]:
    p = (PROJECT / path).resolve()
    if not p.is_relative_to(PROJECT.resolve()):
        return False, f"路径越界：{path} 解析后落在 {p}，位于项目根之外"
    return True, str(p)

print(check_path("../project_secret/config.ini"))
print(check_path("src/main.py"))

**防呆二：edit_file 的唯一匹配。** `old_text` 必须在文件中恰好出现 1 次；出现 0 次或多次时，错误信息要写清"为什么失败、怎么改"，让模型下一轮自我修正（书 4.3：whitespace mismatch 属于可自我修正错误）。

In [ ]:
# ===== Demo 3：edit_file 防呆——唯一匹配与"可自我修正"的错误 =====
"""old_text 必须恰好出现 1 次（书 4.3）。
失败信息写清"为什么失败 + 怎么改"，让模型下一轮自己修正：
错误是给模型的反馈信号，不是异常。
"""
def edit_file(content, old_text, new_text):
    count = content.count(old_text)
    if count == 0:
        return False, ("old_text 未找到：检查空白与缩进是否与文件一致\n"
                       "提示：先 read_file 拿到原文，再从中复制 old_text")
    if count > 1:
        lines = [i + 1 for i, ln in enumerate(content.splitlines()) if old_text in ln]
        return False, f"old_text 出现 {count} 次（行 {lines}），请扩大上下文使其唯一"
    return True, content.replace(old_text, new_text, 1)

doc = "def main():\n    print('hi')\n    print('hi')\n"
ok, msg = edit_file(doc, "print('hi')", "print('hello')")
print(ok, "|", msg)
ok, msg = edit_file(doc, "    print('hi')\n    print('hi')", "    print('hello')")
print(ok, "|", repr(msg))
ok, msg = edit_file(doc, "print('bye')", "x")
print(ok, "|", msg)

## 3. 描述工程实验（表 4-3）：v1 → v2 只做三件事

**Golden Set**：100 个任务三分——positive 61 / 模糊 32 / negative 7；4 组合 × 100 任务 × 3 seeds = **1200 次运行**。

v2 工具描述（555 字符 / 146 Token，c110k_base 计，约为 v1 的 5–8 倍）只改三处：

1. **负向约束前置**：最占 Token 的"不要做"放在描述最前面；
2. **英文化**：工具协议语言对齐模型训练分布；
3. **NEVER 指向替代工具**：不说"禁止 X"，而说"NEVER use X; use Y instead"。

v2 系统提示词（约 70 行）三条规则：概念性问题直接回答、不调用工具；Shell 需求翻译成专用工具调用；响应前先声明意图。

| 组合 | 首次调用准确率 | 任务命中率 | 违规调用率 | bash_positive 首次命中率 | edit_positive 首次命中率 | edit_positive 任务内命中率 |
|---|---|---|---|---|---|---|
| v1 描述 + v1 提示词（基线） | 60.0% | 88.3% | 13.0% | 87.5% | 0% | 76.7% |
| v2 描述 + v1 提示词 | 66.3% | 90.0% | 5.7% | 54.2% | 0% | 80.0% |
| v1 描述 + v2 提示词 | 59.3% | 89.3% | 13.7% | 83.3% | 0% | 83.3% |
| v2 描述 + v2 提示词 | 65.7% | 89.3% | **5.3%** | 54.2% | 0% | 73.3% |

**指标陷阱**：edit_positive 的"首次命中率"四组全是 0%——不是 edit 不可用，而是编辑任务天然要先 read 再 edit；换成"任务内命中率"口径立刻回到 73.3%–83.3%。只看单一指标会得出"edit 完全失效"的错误结论。

## 4. 描述长度的"倒U"（表 4-4）：过约束反噬

8 档描述长度 × 23 个 bash 任务 × 3 seeds = **552 次调用**。准确率不是单调递增，而是倒U：

| Token | 首次调用准确率 | 违规调用率 | bash 成功使用率 |
|---|---|---|---|
| 15 | 65.2% | 30.4% | 87.5% |
| 45 | 85.5% | 13.0% | 95.8% |
| 80 | 78.3% | 18.8% | 91.7% |
| 146 | 79.7% | 10.1% | 70.8% |
| 303 | **97.1%** | 0.0% | 91.7% |
| 521 | **97.1%** | 1.4% | 95.8% |
| 761 | 88.4% | 5.8% | 83.3% |
| 1172 | 84.1% | 7.2% | 75.0% |

<div style="text-align:center"><img src="../src/tool_description_curve.svg" width="820"/></div>

- **太短**（15 Token）：违规率飙到 30.4%——描述不足等于放任模型乱调；
- **甜点区**（303–521 Token）：双峰 97.1%，违规率几乎归零；
- **过长**（>521 Token）：准确率 88.4% → 84.1%，bash 成功使用率 95.8% → 75.0%——约束过度，模型开始"不敢用"。

In [ ]:
# ===== Demo 4：表4-4 复现——找峰值与"过约束"反拐点 =====
"""数据来自书 4.6 表 4-4：8 档描述长度 × 23 个 bash 任务 × 3 seeds = 552 次调用。
脚本做三件事：找峰值、找反拐点、量化"太短"与"太长"两端的损失。
"""
rows = [
    # (Token, 首次调用准确率%, 违规调用率%, bash 成功使用率%)
    (15, 65.2, 30.4, 87.5), (45, 85.5, 13.0, 95.8), (80, 78.3, 18.8, 91.7),
    (146, 79.7, 10.1, 70.8), (303, 97.1, 0.0, 91.7), (521, 97.1, 1.4, 95.8),
    (761, 88.4, 5.8, 83.3), (1172, 84.1, 7.2, 75.0),
]
best = max(rows, key=lambda r: (r[1], -r[2]))
print(f"峰值：{best[0]} Token → 准确率 {best[1]}%（303 与 521 双峰并列）")

peak_idx = rows.index(best)
for tok, acc, viol, _ in rows[peak_idx + 1:]:
    if acc < best[1] - 5:
        print(f"反拐点：{tok} Token → 准确率跌至 {acc}%、违规回升至 {viol}%，进入'过约束'区")
        break

short, long = rows[0], rows[-1]
print(f"太短（{short[0]} Token）：准确率 {short[1]}%、违规 {short[2]}% —— 描述不足=放任模型乱调")
print(f"太长（{long[0]} Token）：准确率 {long[1]}%、bash 使用率 {long[3]}% —— 约束过度=模型不敢用")

## 5. 负协同（4.6.4）：局部最优 ≠ 全局最优

把各自单独验证过的优化直接叠加，可能互相拉扯（书 4.6.4）：

- v1 描述 + v1 提示词：bash_positive 首次命中率 **87.5%**
- 换上 v2 描述后（无论提示词配 v1 还是 v2）：跌到 **54.2%**
- v2 描述 + v2 提示词的"双优化"组合，也没有把 bash 首次命中率救回来

两份"各自变好"的文本叠在一起反而变差——描述的强约束与提示词的"先声明意图"在 bash 场景互相干扰。教训：**优化必须以组合为单位跑完整 Golden Set，而不是各模块单独验收后直接合并**。

## 6. MCP：把"实现通道"标准化

MCP（Model Context Protocol，Anthropic 2024 年 11 月开源）解决的是三层架构里"通道层"的碎片化：任何工具提供方只要实现 MCP Server，就能被任何 Agent Host 使用。

- **数据层 = JSON-RPC 2.0**（spec 2025-06-18）：方法调用 + 通知两类消息；
- **握手三步**：`initialize`（带 `protocolVersion` / `capabilities` / `clientInfo`）→ 服务端响应并声明能力 → 客户端回 `notifications/initialized` 通知；
- **工具发现**：`tools/list`，返回的 `inputSchema` 与 OpenAI function schema **同构**，可无缝混用；
- **工具调用**：`tools/call`；服务端工具列表变化时推 `notifications/tools/list_changed`；
- **两类错误分开**：工具业务失败走 `result.isError: true`；协议级错误走 JSON-RPC `error`（如 `-32601 method not found`）；
- **传输层**：stdio（换行分隔 JSON）用于本地子进程；Streamable HTTP（POST + SSE，OAuth 鉴权）用于远程。

<div style="text-align:center"><img src="../src/mcp_sequence.svg" width="820"/></div>

对模型完全透明：注册表把 MCP Server 的 `tools/list` 拼进工具列表、把 `tools/call` 的结果转回 `tuple[bool, str]`——**模型永远不知道工具由谁实现**。

In [ ]:
# ===== Demo 5：MCP 时序模拟——initialize → tools/list → tools/call =====
"""书 4.4：MCP 数据层 = JSON-RPC 2.0。三个关键点：
①握手三步（initialize → 能力声明 → notifications/initialized）
②tools/list 的 inputSchema 与 OpenAI function schema 同构
③两类错误分开：工具业务失败用 isError:true，协议错误走 JSON-RPC error
"""
import json

def rpc(id_=None, **kw):
    msg = {"jsonrpc": "2.0", **kw}
    if id_ is not None:
        msg["id"] = id_
    return msg

# ① 握手三步
req = rpc(1, method="initialize", params={
    "protocolVersion": "2025-06-18",
    "capabilities": {}, "clientInfo": {"name": "mini-host", "version": "0.1"}})
print("C->S:", json.dumps(req, ensure_ascii=False)[:80], "...")
resp = rpc(1, result={"protocolVersion": "2025-06-18", "capabilities": {"tools": {}}})
print("S->C:", json.dumps(resp, ensure_ascii=False)[:80], "...")
notify = rpc(method="notifications/initialized")
print("C->S:", notify, "  <- 通知：无 id，无需响应")

# ② tools/list：inputSchema 与 OpenAI function schema 同构
tool = {"name": "read_file", "description": "读取文本文件",
        "inputSchema": {"type": "object",
                        "properties": {"path": {"type": "string"}},
                        "required": ["path"]}}
print("tools/list ->", json.dumps({"tools": [tool]}, ensure_ascii=False)[:90], "...")

# ③ 两类错误分开：业务失败仍走 result（isError），协议错误走 error
def tools_call(name, args):
    if name != "read_file":
        return rpc(3, result={"content": [{"type": "text", "text": f"未知工具 {name}"}],
                              "isError": True})
    return rpc(3, result={"content": [{"type": "text", "text": "文件内容..."}],
                          "isError": False})

print("工具失败:", json.dumps(tools_call("nope", {}), ensure_ascii=False))
print("协议错误:", json.dumps(rpc(9, error={"code": -32601, "message": "method not found"}),
                              ensure_ascii=False))

## 7. 评测基础设施（4.6.5）：先建基线，再做优化

4.6 的所有结论都建立在一套轻量评测设施上：

1. **Capture-only 框架**：不干预 Agent 运行，只记录每次工具调用（任务、seed、调用序列、结果）；
2. **双轨指标报告生成器**：首次命中率与任务内命中率并排输出，杜绝口径误导；
3. **回归基线**：任何描述/提示词改动，先跑同一 Golden Set 对比基线，再决定是否合并。

In [ ]:
# ===== Demo 6：双轨指标报告——同一批运行，两种口径天差地别 =====
"""书 4.6.5：edit_positive 的"首次命中率 0%"与"任务内命中率 73.3%–83.3%"
是同一批运行的两种统计口径。只报一种=误导；报告生成器必须双轨并排。
下面用 6 条模拟捕获记录演示口径差异。
"""
from dataclasses import dataclass

@dataclass
class Capture:
    task: str
    seed: int
    calls: list    # 按顺序记录的每次工具调用名
    solved: bool   # 任务最终是否完成

runs = [
    Capture("edit-a", 1, ["bash", "edit_file", "edit_file"], True),
    Capture("edit-a", 2, ["bash", "edit_file"], True),
    Capture("edit-a", 3, ["read_file", "bash"], True),          # 用 bash 改写完成
    Capture("edit-b", 1, ["bash", "bash"], False),
    Capture("edit-b", 2, ["grep_search", "bash", "edit_file"], True),
    Capture("edit-b", 3, ["bash", "edit_file"], True),
]

def first_call_rate(rs, tool):
    hits = sum(1 for r in rs if r.calls and r.calls[0] == tool)
    return hits / len(rs)

def in_task_rate(rs, tool):
    solved = [r for r in rs if r.solved]
    hits = sum(1 for r in solved if tool in r.calls)
    return hits / len(solved) if solved else 0.0

t = "edit_file"
print(f"{t}: 首次命中率 {first_call_rate(runs, t):.1%} | 任务内命中率 {in_task_rate(runs, t):.1%}")
print("结论：edit 天然排在 read/bash 之后，'首次命中率'恒低是口径问题，不是能力问题")

## 8. 工业界印证

| 事实 | 说明 |
|---|---|
| MCP 成为事实标准 | Anthropic 2024-11 开源；2025-03 OpenAI 宣布支持；2025-04 Google DeepMind 宣布 Gemini 支持 |
| 工具描述即产品 | 主流模型厂商的工具调用规范都强调"描述写给模型读"——与表 4-3 / 4-4 的实验结论一致 |
| 评测驱动开发 | 生产 Agent 团队普遍维护工具调用回归基线，任何描述/提示词改动都过评测 |

## 9. 如何做：渐进式落地清单

| 阶段 | 做什么 | 对应 |
|---|---|---|
| L0 | 6 个基础工具 + 统一契约 `tuple[bool, str]`，异常全量兜底 | Demo 1–3 |
| L1 | 注册表收口：白名单 + 阶段权限 + 四处防呆 | Demo 1、2、3 |
| L2 | 描述工程 + 双轨评测：负向约束前置、NEVER 指向替代工具，建立回归基线 | Demo 4、6，表 4-3 / 4-4 |
| L3 | MCP 接入外部工具生态，实现通道与 Agent 解耦 | Demo 5 |

## 10. 核心结论

1. 模型只面对"描述"，执行全部收口在注册表层——描述决定会不会用，契约决定安不安全。
2. 统一契约 `tuple[bool, str]`：错误是写给模型的反馈信号，不是异常；防呆做在工具内，不指望模型自觉。
3. 描述工程有明确收益（违规率 13.0% → 5.3%，准确率 60.0% → 66.3%），但存在"倒U"与"负协同"——有效、有界、需整体评测。
4. 单一指标必然误导：edit 首次命中率 0% vs 任务内 73.3%–83.3%，双轨报告是底线。
5. MCP 用 JSON-RPC 2.0 把实现通道标准化：`tools/list` 与 function schema 同构、两类错误分开、传输层 stdio / Streamable HTTP 分工明确。
6. 顺序不能反：先建 Capture-only 评测设施与回归基线，再做任何描述/提示词优化——没有基线的优化是赌博。

## 参考来源

- 《Harness Engineering 实战：构建可靠的生产级 AI Agent》第 4 章"工具系统与 MCP"，作者陆徐洲（Lex）
- 配套仓库 [harness-py-book](https://github.com/Luxuzhou/harness-py-book)：`harness_py/tools.py`（注册表与 6 个工具）、`harness_py/mcp_client.py`（MCP 客户端）、`examples/ch04_tools.py`（第 4 章入口，免 API Key 运行）
- MCP 规范（2025-06-18 版）：https://modelcontextprotocol.io
- 本书 226 项测试保证代码与文字行为一致